# DAY 1

In [1]:
import pdfplumber

pdf_path = "Training RAG.pdf"

In [2]:
all_text = ""

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            all_text += page_text 

In [3]:
# Print first 500 characters
print("First 500 characters: ")
print(all_text[:500],"\n")

# Last page text
print(page_text)

First 500 characters: 
UNIT I
Introduction to Machine Learning
1. Introduction
1.1 What Is Machine Learning?
Machine learning is programming computers to optimize a performance criterion using example
data or past experience. We have a model defined up to some parameters, and learning is the
execution of a computer program to optimize the parameters of the model using the training data or
past experience. The model may be predictive to make predictions in the future, or descriptive to gain
knowledge from data, or both 

• Add to G all minimal specializations h of g such that
• h is consistent with d, and some member of S is more specific than h
• Remove from G any hypothesis that is less general than another hypothesis in G
CANDIDATE- ELIMINTION algorithm using version spaces
1.7.4 An Illustrative Example
Example Sky AirTemp Humidity Wind Water Forecast EnjoySport
1 Sunny Warm Normal Strong Warm Same Yes
2 Sunny Warm High Strong Warm Same Yes
3 Rainy Cold High Strong Warm Change No
4 S

In [4]:
# Print total character's count
print(f"Total character count: {len(all_text)}")

Total character count: 42536


# DAY 2 & 3- CHUNKING

**1. Character-based sliding window CHUNKING**

In [5]:
CHUNK_SIZE = 2500 # ~ 600 tokens
overlap = 500

chunks = []
start = 0

while start < len(all_text):
    end = start + CHUNK_SIZE
    chunk = all_text[start:end]
    chunks.append(chunk)
    start = end - overlap

print(f"Number of chunks: {len(chunks)}")
print(f"FIRST CHUNK: {chunks[0:1000]}")
print(f"LAST CHUNK: {chunks[-1:1000]}")

Number of chunks: 22
FIRST CHUNK: ['UNIT I\nIntroduction to Machine Learning\n1. Introduction\n1.1 What Is Machine Learning?\nMachine learning is programming computers to optimize a performance criterion using example\ndata or past experience. We have a model defined up to some parameters, and learning is the\nexecution of a computer program to optimize the parameters of the model using the training data or\npast experience. The model may be predictive to make predictions in the future, or descriptive to gain\nknowledge from data, or both.\nArthur Samuel, an early American leader in the field of computer gaming and artificial intelligence,\ncoined the term “Machine Learning” in 1959 while at IBM. He defined machine learning as “the field of\nstudy that gives computers the ability to learn without being explicitly programmed.” However, there is\nno universally accepted definition for machine learning. Different authors define the term differently.\nDefinition of learning\nDefinition\nA 

**2. Section based CHUNKING (Dependency - aware)**

In [6]:
# Step 1: Identify seciton boundaries
import re
section_patter = r"\n\d+\.\d+.*" # pattern for section headers like 1.1, 1.2, 2.3 etc.

# Step 2: Split text by these boundaries
sections = re.split(section_patter, all_text)  # Now sections is a list

# Step 3: Clean and store meaningful chunks
chunks = []

for section in sections:
    cleaned = section.strip()
    if len(cleaned) > 300:
        chunks.append(cleaned)

# Step 4: Inspect results
print("Number of chunks:", len(chunks))

print("\nFIRST CHUNK:\n")
print(chunks[0][:1000])

print("\nLAST CHUNK:\n")
print(chunks[-1][:1000])

Number of chunks: 22

FIRST CHUNK:

Machine learning is programming computers to optimize a performance criterion using example
data or past experience. We have a model defined up to some parameters, and learning is the
execution of a computer program to optimize the parameters of the model using the training data or
past experience. The model may be predictive to make predictions in the future, or descriptive to gain
knowledge from data, or both.
Arthur Samuel, an early American leader in the field of computer gaming and artificial intelligence,
coined the term “Machine Learning” in 1959 while at IBM. He defined machine learning as “the field of
study that gives computers the ability to learn without being explicitly programmed.” However, there is
no universally accepted definition for machine learning. Different authors define the term differently.
Definition of learning
Definition
A computer program is said to learn from experience E with respect to some class of tasks T and
perform

**3. Section based CHUNKING (independent - aware)**

In [7]:
# Step 1: Detect "example" blocks
def is_example_section(text):
    keywords = ["example","illustration","table"]
    text_lower = text.lower()
    return any (k in text_lower for k in keywords)

# Step 2: Merge examples with previous explanation
merged_chunks = []

for section in chunks: # section-based chunks (dependency)
    if is_example_section(section) and merged_chunks:
        merged_chunks[:-1] += "\n\n" + section
    else:
        merged_chunks.append(section)

# Step 3: Now control size

# Apply secondary split
MAX_CHARS = 3000
final_chunks = []

for chunk in merged_chunks:
    if len(chunk) <= MAX_CHARS:
        final_chunks.append(chunk)
    else : # Split internally
        start = 0
        while start < len(chunk):
            end = start + MAX_CHARS
            final_chunks.append(chunk[start:end])
            start = end - 400


# Step 4: Inspect results
print("Number of chunks:", len(chunks))

print("\nLAST CHUNK:\n")
print(chunks[-1][:1000])

Number of chunks: 22

LAST CHUNK:

Example Sky AirTemp Humidity Wind Water Forecast EnjoySport
1 Sunny Warm Normal Strong Warm Same Yes
2 Sunny Warm High Strong Warm Same Yes
3 Rainy Cold High Strong Warm Change No
4 Sunny Warm High Strong Cool Change Yes
CANDIDATE-ELIMINTION algorithm begins by initializing the version space to the set of all
hypotheses in H;
Initializing the G boundary set to contain the most general hypothesis in H
G0 ?, ?, ?, ?, ?, ?
Initializing the S boundary set to contain the most specific (least general) hypothesis
S0 , , ,  , , 
 When the first training example is presented, the CANDIDATE-ELIMINTION algorithm checks the
S boundary and finds that it is overly specific and it fails to cover the positive example.
 The boundary is therefore revised by moving it to the least more general hypothesis that covers
this new example
 No update of the G boundary is needed in response to this training example because Go
correctly covers this example
 When th

In [8]:
def is_continuation(text):
    text = text.strip()
    
    if text.startswith(("•", "-", "")):
        return True
    
    continuation_phrases = [
        "this example",
        "the second training",
        "therefore",
        "hence",
        "as presented",
        "leaving",
        "unchanged"
    ]
    
    lower = text.lower()
    return any(p in lower for p in continuation_phrases)

# Merge continuation backwards
final_chunks = []

for chunk in merged_chunks:
    if final_chunks and is_continuation(chunk):
        final_chunks[-1] += "\n\n" + chunk
    else:
        final_chunks.append(chunk)

print("Number of chunks:", len(chunks)) #22 chunks

print("\nLAST CHUNK:\n")
print(chunks[21])

Number of chunks: 22

LAST CHUNK:

Example Sky AirTemp Humidity Wind Water Forecast EnjoySport
1 Sunny Warm Normal Strong Warm Same Yes
2 Sunny Warm High Strong Warm Same Yes
3 Rainy Cold High Strong Warm Change No
4 Sunny Warm High Strong Cool Change Yes
CANDIDATE-ELIMINTION algorithm begins by initializing the version space to the set of all
hypotheses in H;
Initializing the G boundary set to contain the most general hypothesis in H
G0 ?, ?, ?, ?, ?, ?
Initializing the S boundary set to contain the most specific (least general) hypothesis
S0 , , ,  , , 
 When the first training example is presented, the CANDIDATE-ELIMINTION algorithm checks the
S boundary and finds that it is overly specific and it fails to cover the positive example.
 The boundary is therefore revised by moving it to the least more general hypothesis that covers
this new example
 No update of the G boundary is needed in response to this training example because Go
correctly covers this example
 When th

# DAY 4 & 5 - EMBEDDINGS

**Step 1: Load API key for embeddings**

In [9]:
from openai import OpenAI

with open("api_key.txt", "r") as f:
    api_key = f.read().strip()

client = OpenAI(api_key= api_key)

embeddings = []

for chunk in chunks:

    if not isinstance(chunk, str):
        continue
    if not chunk.strip():
        continue

    response = client.embeddings.create(
        model= "text-embedding-3-small",
        input = chunk
    )

    vector = response.data[0].embedding
    embeddings.append(vector)

**Step 2: Cosine Similarity**

In [10]:
import math

def cosine_similarity(vec1, vec2):
    dot_product = sum(a*b for a,b in zip(vec1, vec2))
    norm1 = math.sqrt(sum (a*a for a in vec1))
    norm2 = math.sqrt(sum (b*b for b in vec2))
    return dot_product/(norm1 * norm2)

**Step 3: Embed the user query**

In [11]:
query = "Explain candidate elimination Algorithm"

query_embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
).data[0].embedding

In [12]:
print(query_embedding)

[0.013930964283645153, -0.01695280522108078, 0.022234095260500908, -0.026406453922390938, -0.0027983218897134066, -0.004976334515959024, 0.014159681275486946, 0.05361688509583473, -0.03689972683787346, 0.06858747452497482, 0.02855501137673855, -0.06193387880921364, -0.02691933512687683, 0.014021065086126328, 0.05533573031425476, 0.018075598403811455, 0.017202315852046013, -0.018810266628861427, -0.05142674222588539, 0.05960512161254883, 0.025186628103256226, 0.024756917729973793, 0.01966968923807144, 0.036733388900756836, 0.04518899694085121, -0.020473666489124298, 0.06930828094482422, -0.006608544383198023, 0.02872135117650032, -0.01962810568511486, 0.01757657900452614, -0.013840863481163979, -0.02337075211107731, -0.036872003227472305, -0.0066431984305381775, 0.039200764149427414, 0.01571911759674549, 0.04225032776594162, -0.028471840545535088, -0.000814372266177088, -0.0041065155528485775, 0.021222194656729698, -0.04804449900984764, 0.03592941164970398, 0.03678883612155914, 0.048654

In [13]:
print(embeddings[:5])

[[-0.030943386256694794, -0.03314371407032013, 0.001547417021356523, -0.03591890633106232, 0.06331401318311691, 0.017533263191580772, 0.03377804532647133, 0.010942183434963226, 0.016512390226125717, 0.08349362015724182, 0.03421414643526077, 0.0013838789891451597, -0.043729089200496674, -0.031161436811089516, 0.027434751391410828, 0.04053761810064316, 0.01298393215984106, 0.0032781949266791344, 0.06339330226182938, 0.012617209926247597, 0.0413503535091877, 0.035344045609235764, 0.045156329870224, 0.02293497510254383, 0.029139511287212372, -0.032886020839214325, 0.007666466757655144, 0.026681484654545784, 0.002026880858466029, -0.026760775595903397, 0.005381889175623655, 0.0013714897213503718, -0.0381787046790123, -0.010823247022926807, 0.02602733112871647, -0.016988135874271393, 0.02983330935239792, -0.02134915255010128, -0.018187416717410088, 0.018920859321951866, -0.05031025782227516, -0.022835861891508102, -0.009713170118629932, 0.07080703228712082, -0.02656254731118679, -0.045077040

**Step 4: Compute similarity with ALL chunks**

In [14]:
scores = []

for i, chunk_embedding in enumerate(embeddings):
    score = cosine_similarity(query_embedding, chunk_embedding)
    scores.append((score, i))

In [15]:
len(query_embedding) == len(embeddings[0])


True

**Step 5: Retrieve TOP-K (This is retrival)**

In [16]:
scores.sort(reverse=True)

top_k = 3
top_chunks = scores[:top_k]

for score, idx in top_chunks:
    print("Score:", round(score, 4))
    print(chunks[idx][:500])
    print("-"*60)

Score: 0.6104
The Candidate – Elimination algorithm finds all describable hypotheses that are consistent with the
15observed training examples. In order to define this algorithm precisely, we begin with a few basic
definitions. First, let us say that a hypothesis is consistent with the training examples if it correctly
classifies these examples.
Definition: A hypothesis h is consistent with a set of training examples D if and only if h(x) = c(x) for
each example (x, c(x)) in D.
Note difference between definit
------------------------------------------------------------
Score: 0.5871
The CANDIDATE-ELIMINTION algorithm computes the version space containing all hypotheses
from H that are consistent with an observed sequence of training examples.
Initialize G to the set of maximally general hypotheses in H Initialize S to the set of maximally specific
hypotheses in H For each training example d, do
• If d is a positive example
• Remove from G any hypothesis inconsistent with d
• For each h

# DAY 6 & 7

**Step 1: Prepare context text**

In [17]:
context = ""

for scores,idx in top_chunks:
    context += chunks[idx]
    context += "\n\n------------\n\n"

print("==== CONTEXT SENT TO LLM ====")
print(context)

==== CONTEXT SENT TO LLM ====
The Candidate – Elimination algorithm finds all describable hypotheses that are consistent with the
15observed training examples. In order to define this algorithm precisely, we begin with a few basic
definitions. First, let us say that a hypothesis is consistent with the training examples if it correctly
classifies these examples.
Definition: A hypothesis h is consistent with a set of training examples D if and only if h(x) = c(x) for
each example (x, c(x)) in D.
Note difference between definitions of consistent and satisfies
 An example x is said to satisfy hypothesis h when h(x) = 1, regardless of whether x is a positive
or negative example of the target concept.
 An example x is said to consistent with hypothesis h iff h(x) = c(x)
Definition: version space- The version space, denoted V SH, D with respect to hypothesis space H and
training examples D, is the subset of hypotheses from H consistent with the training examples in D

------------

The CAND

**Step 2: Call the LLM (low temperature)**

In [18]:
response = client.responses.create(
    model= "gpt-4.1-mini",
    temperature= 0.3,
    max_output_tokens= 300,
    input=[
        {
        "role": "system",
        "content": (
            "You are an assistant that answers ONLY using the provided context. "
            "If the answer is not in the context, reply 'I don't know.' "
            "Do not use prior knowledge. "
        )
        },
        {
            "role":"user",
            "content": f"""
Context:
{context}

Question:
Explain candidate elimination algorithm
"""
        }
    ]
)

print(response.output_text)

The Candidate-Elimination algorithm finds all hypotheses in a hypothesis space H that are consistent with a given set of training examples D. It computes the version space, which is the subset of hypotheses consistent with the observed training data.

The algorithm works as follows:

1. Initialize:
   - G as the set of maximally general hypotheses in H.
   - S as the set of maximally specific hypotheses in H.

2. For each training example d:
   - If d is a positive example:
     - Remove from G any hypothesis inconsistent with d.
     - For each hypothesis s in S that is not consistent with d:
       - Remove s from S.
       - Add to S all minimal generalizations h of s such that:
         - h is consistent with d, and
         - some member of G is more general than h.
       - Remove from S any hypothesis that is more general than another hypothesis in S.
   
   - If d is a negative example:
     - Remove from S any hypothesis inconsistent with d.
     - For each hypothesis g in G t

In [19]:
response = client.responses.create(
    model= "gpt-4.1-mini",
    temperature= 0.3,
    max_output_tokens= 300,
    input=[
        {
        "role": "system",
        "content": (
            "You are an assistant that answers ONLY using the provided context. "
            "If the answer is not in the context, reply 'I don't know.' "
            "Do not use prior knowledge. "
        )
        },
        {
            "role":"user",
            "content": f"""
Context:
{context, "Gradient descent is an optimization algorithm used to minimize a loss function by iteratively moving in the direction of steepest descent. "}

Question:
What is Gradient Descent?
"""
        }
    ]
)

print(response.output_text)

Gradient descent is an optimization algorithm used to minimize a loss function by iteratively moving in the direction of steepest descent.


# DAY 8 & 9 - Using CHORMA DB for RAG

**Create a persistent Chroma collection**

In [1]:
import chromadb
from chromadb.config import Settings # Configuration object for Chroma

chroma_client = chromadb.PersistentClient(
    path = r"C:\Users\user\Documents\Basic_ML\LLMs\chroma_store"
)

collection = chroma_client.get_or_create_collection(name = "ml_textbook")

**Add Chunks**

In [23]:

collection.add(
    ids = [str(i) for i in range(len(chunks))],
    documents = chunks,
    embeddings = embeddings
)

print(collection.count())

22


**Query Chroma**

In [ ]:
from openai import OpenAI

with open("api_key.txt", "r") as f:
    api_key = f.read().strip()

client = OpenAI(api_key= api_key)

# Step 1: Embed the query
query = "Explain Candidate elimination algorithm"

query_embedding = client.embeddings.create(
    model= "text-embedding-3-small",
    input= query
).data[0].embedding

# Step 2: Query Chroma
results = collection.query(
    query_embeddings= [query_embedding], # It should be a list
    n_results= 3
)

retrived_chunks = results["documents"][0]

print(results)

{'ids': [['18', '20', '21']], 'embeddings': None, 'documents': [['The Candidate – Elimination algorithm finds all describable hypotheses that are consistent with the\n15observed training examples. In order to define this algorithm precisely, we begin with a few basic\ndefinitions. First, let us say that a hypothesis is consistent with the training examples if it correctly\nclassifies these examples.\nDefinition: A hypothesis h is consistent with a set of training examples D if and only if h(x) = c(x) for\neach example (x, c(x)) in D.\nNote difference between definitions of consistent and satisfies\n\uf0b7 An example x is said to satisfy hypothesis h when h(x) = 1, regardless of whether x is a positive\nor negative example of the target concept.\n\uf0b7 An example x is said to consistent with hypothesis h iff h(x) = c(x)\nDefinition: version space- The version space, denoted V SH, D with respect to hypothesis space H and\ntraining examples D, is the subset of hypotheses from H consisten